# Energy Demand & Price Analytics - Deep Dive

This notebook connects directly to the warehouse and explores demand, price, and fuel-mix patterns across five U.S. balancing authorities.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from db import get_engine

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('tab10')

engine = get_engine()
print('Connected to warehouse')

In [ ]:
# Load the two fact tables joined to dimensions
demand_sql = '''
SELECT dr.region_code, dt.ts, dt.hour, dt.weekday, dt.is_weekend,
       fed.demand_mw, fed.price_musd_mwh
FROM fact_energy_demand fed
JOIN dim_time dt ON dt.time_id = fed.time_id
JOIN dim_region dr ON dr.region_id = fed.region_id
'''
demand = pd.read_sql(demand_sql, engine, parse_dates=['ts'])

gen_sql = '''
SELECT dr.region_code, dt.ts, dt.hour, df.fuel_type,
       fgm.generation_mwh, fgm.share_percent
FROM fact_generation_mix fgm
JOIN dim_time dt ON dt.time_id = fgm.time_id
JOIN dim_region dr ON dr.region_id = fgm.region_id
JOIN dim_fuel df ON df.fuel_id = fgm.fuel_id
'''
generation = pd.read_sql(gen_sql, engine, parse_dates=['ts'])

print(f'Demand rows: {len(demand):,}')
print(f'Generation rows: {len(generation):,}')

## 1. Average daily demand profile by region

In [ ]:
profile = demand.groupby(['region_code', 'hour'])['demand_mw'].mean().reset_index()
pivot = profile.pivot(index='hour', columns='region_code', values='demand_mw')

fig, ax = plt.subplots(figsize=(10, 5))
pivot.plot(ax=ax, linewidth=2)
ax.set_title('Average Daily Demand Profile by Region')
ax.set_xlabel('Hour of Day (UTC)')
ax.set_ylabel('Demand (MW)')
ax.legend(title='Region')
plt.tight_layout()
plt.show()

## 2. Price heatmap by region and hour

In [ ]:
price_pivot = demand.groupby(['region_code', 'hour'])['price_musd_mwh'].mean().unstack()

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(price_pivot, cmap='YlOrRd', annot=False, fmt='.1f', ax=ax)
ax.set_title('Average Price ($/MWh) by Region and Hour')
ax.set_xlabel('Hour of Day (UTC)')
ax.set_ylabel('Region')
plt.tight_layout()
plt.show()

## 3. Generation mix stacked area (ERCOT sample week)

In [ ]:
ercot_week = generation[(generation['region_code'] == 'ERCO') &
                        (generation['ts'] >= '2024-01-15') &
                        (generation['ts'] < '2024-01-22')]
mix = ercot_week.pivot(index='ts', columns='fuel_type', values='generation_mwh').fillna(0)

fig, ax = plt.subplots(figsize=(12, 5))
ax.stackplot(mix.index, mix.T, labels=mix.columns, alpha=0.85)
ax.set_title('ERCOT Generation Mix - Sample Week (Jan 15-21, 2024)')
ax.set_xlabel('Timestamp (UTC)')
ax.set_ylabel('Generation (MWh)')
ax.legend(loc='upper left', reverse=True)
plt.tight_layout()
plt.show()

## 4. 24-hour moving average vs raw demand (PJM)

In [ ]:
pjm = demand[demand['region_code'] == 'PJM'].sort_values('ts').copy()
pjm['demand_ma_24h'] = pjm['demand_mw'].rolling(window=24, min_periods=24).mean()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(pjm['ts'], pjm['demand_mw'], label='Raw Demand', alpha=0.5)
ax.plot(pjm['ts'], pjm['demand_ma_24h'], label='24h Moving Average', linewidth=2)
ax.set_title('PJM Hourly Demand vs 24-Hour Moving Average')
ax.set_xlabel('Timestamp (UTC)')
ax.set_ylabel('Demand (MW)')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Price spike timeline (top 1% prices per region)

In [ ]:
thresholds = demand.groupby('region_code')['price_musd_mwh'].transform(lambda x: x.quantile(0.99))
spikes = demand[demand['price_musd_mwh'] >= thresholds]

fig, ax = plt.subplots(figsize=(12, 5))
for region, group in spikes.groupby('region_code'):
    ax.scatter(group['ts'], group['price_musd_mwh'], label=region, s=15, alpha=0.7)
ax.set_title('Top 1% Price Hours by Region')
ax.set_xlabel('Timestamp (UTC)')
ax.set_ylabel('Price ($/MWh)')
ax.legend(title='Region')
plt.tight_layout()
plt.show()

## 6. Renewable fuel-mix heatmap (solar + wind share by region and hour)

In [ ]:
renew = generation[generation['fuel_type'].isin(['Solar', 'Wind'])]
renew_mix = renew.groupby(['region_code', 'hour'])['share_percent'].sum().unstack()

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(renew_mix, cmap='Greens', annot=False, fmt='.1f', ax=ax)
ax.set_title('Solar + Wind Share (%) by Region and Hour')
ax.set_xlabel('Hour of Day (UTC)')
ax.set_ylabel('Region')
plt.tight_layout()
plt.show()